### Célula 1: Configuração do AF SDK

Este é o primeiro passo para garantir que o Python consiga se comunicar com o banco de dados da planta. O código injeta as bibliotecas nativas da AVEVA diretamente no ambiente de desenvolvimento. 

**O que você precisa verificar:**
Antes de executar, observe a variável `caminho_sdk`. Ela contém o diretório padrão onde o PI Client costuma ser instalado no Windows (`C:\Program Files (x86)\PIPC\AF\PublicAssemblies\4.0`). Se a instalação corporativa na sua máquina estiver em um diretório diferente, atualize esse caminho. Depois, basta executar a célula para carregar o motor de integração.

In [ ]:
import sys
import clr
import pandas as pd
from datetime import datetime

# Caminho base das bibliotecas nativas do PI Client no Windows
caminho_sdk = r'C:\Program Files (x86)\PIPC\AF\PublicAssemblies\4.0'
sys.path.append(caminho_sdk)

try:
    # Carrega o motor .NET para dentro do Python
    clr.AddReference('OSIsoft.AFSDK')
    print("[SUCESSO] Biblioteca base do AVEVA PI carregada.")
    
except FileNotFoundError as e:
    print("[ERRO] Arquivo OSIsoft.AFSDK.dll não encontrado.")
    print(f"Verifique se o PI Client está instalado no caminho: {caminho_sdk}")
    
except Exception as e:
    print(f"[ERRO] Erro de integração: {e}")

try:
    # Importação das classes estruturais para conexão e busca de dados
    from OSIsoft.AF.PI import PIServers, PIPoint
    from OSIsoft.AF.Time import AFTime, AFTimeRange
    from OSIsoft.AF.Data import AFBoundaryType
    print("[SUCESSO] Componentes estruturais prontos para uso.")
    
except ImportError as e:
    print(f"[ERRO] Falha de importação: {e}")

### Célula 2: Lógica de Extração e Tratamento

Este bloco contém o núcleo de processamento do sistema: a função `extrair_e_salvar_tag_csv`. É ela que gerencia a paginação temporal para não sobrecarregar o servidor e faz a conversão do padrão numérico para evitar erros no Excel.


In [ ]:
def extrair_e_salvar_tag_csv(servidor, nome_da_tag, tempo_inicio_str, tempo_fim_str):
    print(f"\n--- INICIANDO EXTRAÇÃO: {nome_da_tag} ---")
    
    try:
        # Localiza o instrumento no banco de dados
        ponto = PIPoint.FindPIPoint(servidor, nome_da_tag)
        
        if ponto is None:
            print(f"[ERRO] A tag '{nome_da_tag}' não foi encontrada.")
            return
            
        print("Tag localizada com sucesso.")
        
        # Converte o período de busca para o formato do PI Server
        tempo_atual = AFTime(tempo_inicio_str)
        tempo_final = AFTime(tempo_fim_str)
        
        lista_tags = []
        lista_timestamps = []
        lista_valores = []
        
        # Limite máximo para proteção da rede industrial
        LOTE_MAXIMO = 50000
        total_extraido = 0
        numero_lotes = 0
        
        while tempo_atual.UtcTime < tempo_final.UtcTime:
            numero_lotes += 1
            intervalo = AFTimeRange(tempo_atual, tempo_final)
            
            # Requisita os dados brutos armazenados na planta
            dados_lote = ponto.RecordedValues(
                intervalo, AFBoundaryType.Inside, "", False, LOTE_MAXIMO
            )
            
            if dados_lote is None or dados_lote.Count == 0:
                break
            
            for p_dados in dados_lote:
                timestamp_str = p_dados.Timestamp.LocalTime.ToString("yyyy-MM-dd HH:mm:ss")
                valor_bruto = str(p_dados.Value)
                
                # Tratamento para compatibilidade com o Excel brasileiro:
                # Transforma o ponto (padrão PI) em vírgula. 
                # Avisos textuais do sistema são mantidos intactos.
                try:
                    numero = float(valor_bruto.replace(',', '.'))
                    valor_final = str(numero).replace('.', ',')
                except ValueError:
                    valor_final = valor_bruto

                lista_tags.append(nome_da_tag)
                lista_timestamps.append(timestamp_str)
                lista_valores.append(valor_final)
            
            total_extraido += dados_lote.Count
            print(f"   Lote {numero_lotes}: {dados_lote.Count} registros lidos.")
            
            if dados_lote.Count < LOTE_MAXIMO:
                break
            
            # Lógica de Paginação: avança 1ms a partir do último registro para a próxima busca
            ultimo_tempo = dados_lote[dados_lote.Count - 1].Timestamp
            tempo_atual = AFTime(ultimo_tempo.UtcTime.AddMilliseconds(1))
        
        # Estrutura a tabela final apenas com os parâmetros essenciais
        df = pd.DataFrame({
            "Tag": lista_tags,
            "Timestamp": lista_timestamps,
            "Value": lista_valores
        })
        
        print(f"Total: {total_extraido} registros compilados.")
        
        # Salva o arquivo CSV utilizando o ponto-e-vírgula para separar as colunas
        nome_arquivo = f"TAG_{nome_da_tag.replace('.', '_')}.csv"
        df.to_csv(nome_arquivo, index=False, sep=';')
        print(f"Arquivo salvo na pasta local: {nome_arquivo}\n")
        
    except Exception as erro:
        print(f"[ERRO] Ocorreu uma exceção: {erro}")

### Célula 3: Parâmetros e Execução

Esta é a etapa onde você atua como operador do sistema, definindo os parâmetros exatos da extração. Antes de executar o código desta célula, você deve configurar dois pontos fundamentais diretamente no script:

1. **O Intervalo de Tempo:** Defina o período de busca nas variáveis `data_inicio` e `data_fim`. O formato temporal padrão exigido é `AAAA-MM-DD HH:MM:SS` (Ano-Mês-Dia Hora:Minuto:Segundo). Caso deseje extrair os dados até o exato instante da execução, preencha a data final com um asterisco (`*`).
2. **A Lista de Tags:** Na variável `tags_para_buscar`, você deve inserir o nome exato das variáveis de processo que deseja extrair, respeitando rigorosamente a nomenclatura cadastrada no servidor AVEVA PI. Cada tag deve estar entre aspas e separada por vírgula dentro da lista. O sistema processará todas as variáveis listadas sequencialmente, gerando arquivos independentes.

In [ ]:
# Conecta automaticamente ao servidor de dados via credenciais do Windows (SSPI)
servidores = PIServers()
meu_servidor = servidores.DefaultPIServer

print(f"Servidor Alocado: {meu_servidor.Name if meu_servidor else 'Inativo'}")

# Define o período de extração (O caractere '*' representa o instante atual)
data_inicio = "2026-03-01 00:00:00"
data_fim = "*" 

# Insira aqui as tags da planta que deseja extrair
tags_para_buscar = [
    "Process.Plant01.Control.FC101.PV",
    "Process.Plant01.Control.LC101.PV"
]

# Processamento iterativo: executa a extração para cada tag da lista
if meu_servidor:
    for tag in tags_para_buscar:
        extrair_e_salvar_tag_csv(meu_servidor, tag, data_inicio, data_fim)
    print("--- SCRIPT FINALIZADO COM SUCESSO ---")
else:
    print("Erro: Nenhum servidor padrão foi encontrado na rede.")